In [1]:
# CELL 1: Use this cell to manually clear the output directory
import os
import glob
import shutil

# --- WARNING: This will permanently delete the folder and all its contents ---

# Set the path to the directory you want to clear
dir_to_clear = "/scratch/ng72/ms5578/time_series"

# Use shutil.rmtree to remove the entire directory tree
if os.path.exists(dir_to_clear):
    print(f"Directory '{dir_to_clear}' found. Deleting it now...")
    try:
        shutil.rmtree(dir_to_clear)
        print("Directory successfully deleted.")
    except Exception as e:
        print(f"Error deleting directory: {e}")
else:
    print(f"Directory '{dir_to_clear}' does not exist. Nothing to delete.")

# The main processing cell will automatically re-create this directory.

Directory '/scratch/ng72/ms5578/time_series' found. Deleting it now...
Directory successfully deleted.


In [2]:
# === Imports ===
import os
import glob
import re
from datetime import datetime, timedelta
import pandas as pd
from joblib import Parallel, delayed
import fcntl
from tqdm.notebook import tqdm

# === 1. Configuration ===
raw_data_dir = "/g/data/ng72/ms5578/mmsdm_dispatch_data"
write_path = "/scratch/ng72/ms5578/time_series"
start_date = datetime(2009, 7, 1)
end_date = datetime(2024, 7, 1)
N_JOBS = -1

# === 2. SafeWriter Class (No changes needed) ===
class SafeWriter:
    def __init__(self, filepath):
        self.filepath = filepath
        self.file_handle = None
        self.write_header = not os.path.exists(filepath) or os.path.getsize(filepath) == 0
    def __enter__(self):
        self.file_handle = open(self.filepath, 'ab')
        fcntl.flock(self.file_handle, fcntl.LOCK_EX)
        return self
    def append(self, df):
        csv_string = df.to_csv(header=self.write_header, index=False)
        self.file_handle.write(csv_string.encode('utf-8'))
        self.write_header = False
    def __exit__(self, exc_type, exc_val, exc_tb):
        fcntl.flock(self.file_handle, fcntl.LOCK_UN)
        self.file_handle.close()

# === 3. OPTIMIZED Worker Function (with warning fix) ===
def process_single_month(year, month, output_dir):
    FNAME_RE = re.compile(r".*DISPATCHLOAD[_-](\d{12})\.CSV$", flags=re.IGNORECASE)
    NEEDED_COLS = ["SETTLEMENTDATE", "DUID", "INITIALMW", "TOTALCLEARED", "AGCSTATUS"]
    
    start_dt = datetime(year, month, 1)
    end_dt = (start_dt + timedelta(days=32)).replace(day=1)
    all_files = glob.glob(os.path.join(raw_data_dir, "*.CSV")) + glob.glob(os.path.join(raw_data_dir, "*.csv"))
    
    file_list = []
    for f in all_files:
        m = FNAME_RE.match(os.path.basename(f))
        if not m: continue
        try:
            dt = datetime.strptime(m.group(1), "%Y%m%d%H%M")
            if start_dt <= dt < end_dt:
                file_list.append(f)
        except ValueError:
            continue

    if not file_list: return f"⚠️ No files for {year}-{month:02d}."

    frames = []
    for fpath in file_list:
        try:
            df = pd.read_csv(fpath, header=1, usecols=NEEDED_COLS, dtype={"DUID": "category", "INITIALMW": "float32", "TOTALCLEARED": "float32", "AGCSTATUS": "object"}, low_memory=False)
            frames.append(df)
        except Exception as e: return f"❌ Error reading {os.path.basename(fpath)}: {e}"

    if not frames: return f"⚠️ No valid data for {year}-{month:02d}."

    gen_df = pd.concat(frames, ignore_index=True)
    
    gen_df['time'] = pd.to_datetime(gen_df['SETTLEMENTDATE'], format='%Y/%m/%d %H:%M:%S', errors='coerce')
    gen_df = gen_df.drop(columns=['SETTLEMENTDATE'])
    gen_df = gen_df.dropna(subset=['time'])

    gen_df['time_hour'] = gen_df['time'].dt.floor('h')

    gen_df['INITIALMW'] = pd.to_numeric(gen_df['INITIALMW'], errors='coerce').fillna(0.0)
    gen_df['TOTALCLEARED'] = pd.to_numeric(gen_df['TOTALCLEARED'], errors='coerce').fillna(0.0)
    
    gen_df['TOTALMWh'] = gen_df['INITIALMW'] * (5.0 / 60.0)
    gen_df['TOTALCLEARED_MWh'] = gen_df['TOTALCLEARED'] * (5.0 / 60.0)
    
    agg_func = {'TOTALMWh': 'sum', 'TOTALCLEARED_MWh': 'sum', 'AGCSTATUS': 'first'}
    
    grouped = gen_df.groupby(['DUID', 'time_hour'], observed=True).agg(agg_func).reset_index()
    grouped = grouped.rename(columns={'time_hour': 'time'})

    for duid, df_duid in grouped.groupby("DUID", observed=True):
        safe_duid = str(duid).replace("/", "_").replace("\\", "_").replace('#', '_')
        out_path = os.path.join(output_dir, f"{safe_duid}.csv")
        try:
            with SafeWriter(out_path) as writer:
                writer.append(df_duid)
        except Exception as e:
            return f"❌ Failed writing for {safe_duid}: {e}"
            
    # The worker function now returns a simple success/fail message string
    return f"Processed {year}-{month:02d}."

# === 4. FINAL, CORRECTED Main Execution Block ===
if __name__ == '__main__':
    os.makedirs(write_path, exist_ok=True)
    
    all_months = []
    current_date = start_date
    while current_date < end_date:
        all_months.append((current_date.year, current_date.month))
        current_date = (current_date.replace(day=1) + timedelta(days=32)).replace(day=1)
        
    print(f"Starting parallel processing for {len(all_months)} months...")
    
    # Define the list of jobs to be done. The delayed call is now simple and pickle-able.
    tasks = (delayed(process_single_month)(year, month, write_path) for year, month in all_months)
    
    # The Parallel object is an iterable. We wrap it in tqdm.
    # As each job is completed, Parallel yields a result, and tqdm updates the bar.
    results = [
        result for result in tqdm(
            Parallel(n_jobs=N_JOBS, backend="multiprocessing")(tasks), 
            total=len(all_months),
            desc="Processing Months"
        )
    ]

    print("\n--- All chunks processed! ---")
    # You can uncomment the loop below to print the status message from each worker
    # for r in results:
    #     if "⚠️" in r or "❌" in r:
    #         print(r)

Starting parallel processing for 180 months...


Processing Months:   0%|          | 0/180 [00:00<?, ?it/s]


--- All chunks processed! ---


In [4]:
for r in results:
    if "⚠️" in r or "❌" in r:
        print(r)